# Take data from CSV and add to items or make new item

In [ ]:
import pandas as pd
import re
from datetime import datetime
import wikibaseintegrator
from wikibaseintegrator import WikibaseIntegrator, wbi_helpers, wbi_login, datatypes
from wikibaseintegrator.wbi_config import config as wbi_config
from wikibaseintegrator.entities import ItemEntity
from wikibaseintegrator.models import Qualifiers, References, Reference
from wikibaseintegrator.wbi_enums import ActionIfExists
import logging
import json
import random
from pathlib import Path
from typing import Optional, Union
from dataclasses import dataclass, field
from __future__ import annotations
from enum import Enum
import numpy as np

collections_dir = Path("../wikidata/metadata_collections/")

with Path('../authorization.json').open(mode='r') as authorization_file:
    authorization = json.load(authorization_file)

USER_AUTH = authorization['user_auth']
USERNAME = authorization['username']
PASSWORD = authorization['password']
CONSUMER_TOKEN = authorization['consumer_token']
CONSUMER_SECRET = authorization['consumer_secret']

BOTNAME = authorization['botname']
    

logging.basicConfig(filename='wikibaseint-debug.log', 
                    format='%(asctime)s %(message)s', 
                    datefmt='%Y/%m/%d %H:%M:%S',
                    encoding='utf-8', 
                    level=logging.DEBUG)

logger = logging.getLogger('Make-Items')
logger.debug('Start logging')


wbi_config['USER_AGENT'] = f'{BOTNAME} (https://www.wikidata.org/wiki/User:{USERNAME})'
wbi_config['MEDIAWIKI_API_URL'] = 'https://test.wikidata.org/w/api.php'

PROPS = {'instance_of':'P31',
        'author':'P50',
         'title':'P1476',
         'has_edition':'P747',
         'edition_of': 'P629',
         'based_on' : 'P144',
         'language':'P407',
         'publication_date':'P577',
         'work_available_at_URL':'P953',
         'has_edition_or_translation' : 'P747',
         'project_gb_ebook_id' : 'P2034',
         }
ENTITIES = {
   'literary_work':'Q7725634', 
   'edition' : 'Q3331189',
   'German':'Q188',
   'Hugo_Ball':'Q70989'
}

## Load data from CSV (containing data from Openrefine)

In [115]:
fiction = pd.read_csv(Path(collections_dir, 'de_fiction_metadata_2025-11-14T18.csv'), index_col=0, dtype={'gutenberg_id':str})
fiction

,author,author_last,author_first,title,work_qid,gutenberg_id,year,url,pgde_author_id,author_qid,...,filename,author_gender,num_sents,num_tokens,source,work_qid_by,pgde_qid_by,pgus_qid_by,ed1_qid_by,1e_qid_by
index,,,,,,,,,,,,,,,,,,,,,
0,Arthur Achleitner,Achleitner,Arthur,Der Finanzer,NaN,NaN,1916,https://www.projekt-gutenberg.org/achleitn/fin...,achleitn,Q77439,...,Arthur_Achleitner_-_Der_Finanzer.txt,m,1723,24034,PG-DE,NaN,NaN,NaN,NaN,NaN
1,Arthur Achleitner,Achleitner,Arthur,Das Schloß im Moor,NaN,NaN,1903,https://www.projekt-gutenberg.org/achleitn/moo...,achleitn,Q77439,...,Arthur_Achleitner_-_Das_Schloß_im_Moor.txt,m,3460,51072,PG-DE,NaN,NaN,NaN,NaN,NaN
2,Arthur Achleitner,Achleitner,Arthur,Familie Lugmüller,NaN,NaN,1896,https://www.projekt-gutenberg.org/achleitn/lug...,achleitn,Q77439,...,Arthur_Achleitner_-_Familie_Lugmüller.txt,m,1967,26741,PG-DE,NaN,NaN,NaN,NaN,NaN
3,Arthur Achleitner,Achleitner,Arthur,Der Bezirkshauptmann. Erster Teil,NaN,NaN,1901,https://www.projekt-gutenberg.org/achleitn/bez...,achleitn,Q77439,...,Arthur_Achleitner_-_Der_Bezirkshauptmann._Erst...,m,2614,34895,PG-DE,NaN,NaN,NaN,NaN,NaN
4,Arthur Achleitner,Achleitner,Arthur,Geschichten aus den Bergen,NaN,NaN,1910,https://www.projekt-gutenberg.org/achleitn/ber...,achleitn,Q77439,...,Arthur_Achleitner_-_Geschichten_aus_den_Bergen...,m,6350,127192,PG-DE,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4229,Arnold Zweig,Zweig,Arnold,Die Novellen um Claudia,NaN,52478,1912,NaN,NaN,NaN,...,"Zweig,_Arnold_-_Die_Novellen_um_Claudia-52478.txt",m,2720,47335,PG-US,NaN,NaN,NaN,NaN,NaN
4230,Friderike Maria Burger Winternitz Zweig,Zweig,Friderike Maria Burger Winternitz,Vögelchen,NaN,57114,1919,NaN,NaN,NaN,...,"Zweig,_Friderike_Maria_Burger_Winternitz_-_Vög...",m,7846,104521,PG-US,NaN,NaN,NaN,NaN,NaN
4231,Stefan Zweig,Zweig,Stefan,Amok: Novellen einer Leidenschaft,Q675609,57850,1922,NaN,zweig,Q78491,...,"Zweig,_Stefan_-_Amok:_Novellen_einer_Leidensch...",m,3087,69716,PG-US,NaN,NaN,NaN,NaN,NaN


## Log in to Wikidata

In [ ]:
login_instance = wbi_login.Login(user=USER_AUTH, password=PASSWORD)

wbi = WikibaseIntegrator(login=login_instance)

randhex = "{:x}".format(random.randrange(0, 2**48))
EDIT_SUMMARY=f'{BOTNAME}: test ([[:toolforge:editgroups/b/CB/{randhex}|details]])'

# Start making new Entities

In [ ]:
def make_sparql_authors_works(author_qid):
    return '''SELECT ?item ?title ?year (COUNT(?edition) as ?count)
WHERE {
    # is a literary work
  ?item wdt:P31 wd:Q7725634 .
  # author is ...
  ?item wdt:P50 wd:''' + author_qid + ''' . 
  OPTIONAL {
  ?edition wdt:P629 ?item . }
  OPTIONAL {
    ?item wdt:P1476 ?title .} 
  OPTIONAL {
    ?item wdt:P577 ?year . }
  }
GROUP BY ?item ?title ?year
ORDER BY DESC(?count)'''




In [116]:
hennings_qid = 'Q76815'
works_by_Hennings = wbi_helpers.execute_sparql_query(make_sparql_authors_works(hennings_qid), 
                                          user_agent=wbi_config['USER_AGENT'])
works_by_Hennings

{'head': {'vars': ['item', 'title', 'year', 'count']},
 'results': {'bindings': [{'item': {'type': 'uri',
     'value': 'http://www.wikidata.org/entity/Q136794361'},
    'title': {'xml:lang': 'de', 'type': 'literal', 'value': 'Gefängnis'},
    'year': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
     'type': 'literal',
     'value': '1919-01-01T00:00:00Z'},
    'count': {'datatype': 'http://www.w3.org/2001/XMLSchema#integer',
     'type': 'literal',
     'value': '2'}}]}}

## Let's upload our data for Emmy Hennings

In [117]:
view = fiction[fiction['author'].str.contains("Hennings")]
view

,author,author_last,author_first,title,work_qid,gutenberg_id,year,url,pgde_author_id,author_qid,...,filename,author_gender,num_sents,num_tokens,source,work_qid_by,pgde_qid_by,pgus_qid_by,ed1_qid_by,1e_qid_by
index,,,,,,,,,,,,,,,,,,,,,
1396,Emmy Hennings,Hennings,Emmy,Gefängnis,NaN,NaN,1919,https://www.projekt-gutenberg.org/hennings/gef...,hennings,Q76815,...,Emmy_Hennings_-_Gefängnis.txt,f,3808,33814,PG-DE,NaN,NaN,NaN,NaN,NaN
3862,Emmy Ball-Hennings,Ball-Hennings,Emmy,Die letzte Freude,NaN,40218,1913,NaN,NaN,Q76815,...,"Ball-Hennings,_Emmy_-_Die_letzte_Freude-40218.txt",f,83,783,PG-US,NaN,NaN,NaN,NaN,NaN


In [118]:
author_qid = hennings_qid
author_entity = wbi.item.get(entity_id=author_qid)
author_entity.get_json()

{'labels': {'arz': {'language': 'arz', 'value': 'امى هينينجس'},
  'ast': {'language': 'ast', 'value': 'Emmy Hennings'},
  'be-tarask': {'language': 'be-tarask', 'value': 'Эмі Генінгс'},
  'ca': {'language': 'ca', 'value': 'Emmy Hennings'},
  'cs': {'language': 'cs', 'value': 'Emmy Hennings'},
  'cy': {'language': 'cy', 'value': 'Emmy Hennings'},
  'da': {'language': 'da', 'value': 'Emmy Hennings'},
  'de': {'language': 'de', 'value': 'Emmy Hennings'},
  'en': {'language': 'en', 'value': 'Emmy Hennings'},
  'es': {'language': 'es', 'value': 'Emmy Hennings'},
  'eu': {'language': 'eu', 'value': 'Emmy Hennings'},
  'fi': {'language': 'fi', 'value': 'Emmy Hennings'},
  'fr': {'language': 'fr', 'value': 'Emmy Hennings'},
  'ga': {'language': 'ga', 'value': 'Emmy Hennings'},
  'gl': {'language': 'gl', 'value': 'Emmy Hennings'},
  'he': {'language': 'he', 'value': 'אמי הנינגס'},
  'it': {'language': 'it', 'value': 'Emmy Hennings'},
  'ja': {'language': 'ja', 'value': 'エミー・ヘニングス'},
  'nan': {'

In [179]:
def date_tag_precise():
    return datetime.today().strftime("%Y-%m-%dT%H:%M:%S")

class Author():
    def __init__(self, item = None, qid=None, name = None, works = None, editions = None):
        self._item = item
        self._qid = qid
        self._name = name
        self._works = works
        self._editions = editions
    
    @classmethod
    def from_item(self, item: wikibaseintegrator.entities.item.ItemEntity):
        return Author(item=item)
    
    def get_name(self, language='mul'):
        if self._item:
            if self._item.labels.get(language):
                return self._item.labels.get(language).value
            if self._item.labels.get('mul'):
                return self._item.labels.get('mul').value
            if self._name:
                return self._name
            else:
                if self._quid:
                    raise Exception(f'No name available for {self._qid}')
                if self.item:
                    raise Exception(f'No name available for {self._item}')
                raise Exception(f'No name available for {self}')
        else:
            return self.name

            
        

In [180]:
date_tag_precise()

'2025-11-16T00:15:29'

In [120]:
type(author_entity)

wikibaseintegrator.entities.item.ItemEntity

In [121]:
author_entity.labels.get('en').value

'Emmy Hennings'

In [122]:
author = Author.from_item(author_entity)
author.get_name('de')

'Emmy Hennings'

## Make a new work entry

In [123]:
work = view.iloc[1]
work

author                                           Emmy Ball-Hennings
author_last                                           Ball-Hennings
author_first                                                   Emmy
title                                             Die letzte Freude
work_qid                                                        NaN
gutenberg_id                                                  40218
year                                                           1913
url                                                             NaN
pgde_author_id                                                  NaN
author_qid                                                   Q76815
refine_session                                          37/20251114
genre                              Romane, Novellen und Erzählungen
filename          Ball-Hennings,_Emmy_-_Die_letzte_Freude-40218.txt
author_gender                                                     f
num_sents                                       

In [124]:
# CURRENTLY WE ONLY DO SINGLE AUTHORED WORKS
# Otherwise, parsing the names is a hassle and we have only a handful of works with more than one author,
# ...And all of them have data inconsistency issues that need to be adressed first

def format_year(year : int):
    return datetime(year, 1, 1).strftime("+%Y-%m-%dT%H:%M:%SZ")

def make_work(author_qid : str, author : Author, title : str, 
              year : int = None, url : Optional[str] = None): 
    language = ENTITIES['German']

    formatted_year = format_year(year)

    new_work = wbi.item.new()

    new_work.labels.set('de', title)
    # Set a default label too
    new_work.labels.set('mul', title)
    new_work.descriptions.set('en', f'Literary work of fiction by {author.get_name('en')}')
    new_work.descriptions.set('de', f'Fiktionales literarisches Werk von {author.get_name('de')}')
    
    new_work.claims.add([
        datatypes.Item(value=ENTITIES['literary_work'], prop_nr=PROPS['instance_of']), 
        datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
        #
        # NEED TO CHECK IF IT HAS OTHER AUTHORS
        #
        datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
        datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
        datatypes.Item(value=language, prop_nr=PROPS['language'])
                    ])
    if url:
        new_work.claims.add([
            datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                        ])
    return new_work

In [125]:
title = work['title'].strip()
title

'Die letzte Freude'

In [126]:
print(author_name)
author

Emmy Hennings


In [127]:
year = work['year'].item()

# Demonstrate
formatted_year = format_year(year)
formatted_year

'+1913-01-01T00:00:00Z'

In [134]:
url = work['url']
print(url)
print(type(url))
bool(url)
if np.isnan(url):
    url = None
url

nan
<class 'float'>


In [136]:
#author_name = work['author'].strip() 

new_work = make_work(author_qid = author_qid, author=author, title = title, 
                    year = year, url=url)
new_work.get_json()

{'labels': {'de': {'language': 'de', 'value': 'Die letzte Freude'},
  'mul': {'language': 'mul', 'value': 'Die letzte Freude'}},
 'descriptions': {'en': {'language': 'en',
   'value': 'Literary work of fiction by Emmy Hennings'},
  'de': {'language': 'de',
   'value': 'Fiktionales literarisches Werk von Emmy Hennings'}},
 'aliases': {},
 'sitelinks': {},
 'type': 'item',
 'claims': {'P31': [{'mainsnak': {'snaktype': 'value',
     'property': 'P31',
     'datatype': 'wikibase-item',
     'datavalue': {'value': {'entity-type': 'item',
       'numeric-id': 7725634,
       'id': 'Q7725634'},
      'type': 'wikibase-entityid'}},
    'type': 'statement',
    'rank': 'normal'}],
  'P1476': [{'mainsnak': {'snaktype': 'value',
     'property': 'P1476',
     'datatype': 'monolingualtext',
     'datavalue': {'value': {'text': 'Die letzte Freude', 'language': 'de'},
      'type': 'monolingualtext'}},
    'type': 'statement',
    'rank': 'normal'}],
  'P50': [{'mainsnak': {'snaktype': 'value',
    

In [137]:
new_work = new_work.write()
work_qid = new_work.id
work_qid


'Q136795200'

I forgot the url, so let's quickly add that; this is where my own class would be wonderful!

In [ ]:
work_entity = wbi.item.get(entity_id=work_qid)
work_entity.claims.add([
            datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                        ])
work_entity.write()

<ItemEntity @dfb710 _BaseEntity__api=<wikibaseintegrator.wikibaseintegrator.WikibaseIntegrator object at 0x11cdfafc0>
	 _BaseEntity__title='Q136794361'
	 _BaseEntity__pageid=130475019
	 _BaseEntity__lastrevid=2429856840
	 _BaseEntity__type='item'
	 _BaseEntity__id='Q136794361'
	 _BaseEntity__claims=<Claims @df6ff0 _Claims__claims={'P31': [<Item @e06180 _Claim__mainsnak=<Snak @e04800 _Snak__snaktype=<WikibaseSnakType.KNOWN_VALUE: 'value'> _Snak__property_number='P31' _Snak__hash='5963e3cad5eee4ae1cffbefbac15327800b9a7e6' _Snak__datavalue={'value': {'entity-type': 'item', 'numeric-id': 7725634, 'id': 'Q7725634'}, 'type': 'wikibase-entityid'} _Snak__datatype='wikibase-item'> _Claim__type='statement' _Claim__qualifiers=<Qualifiers @e04260 _Qualifiers__qualifiers={}> _Claim__qualifiers_order=[] _Claim__id='Q136794361$6D962434-BE96-4160-98FC-90A0CD3A361F' _Claim__rank=<WikibaseRank.NORMAL: 'normal'> _Claim__removed=False _Claim__references=<References @e073b0 _References__references=[]>>], '

We should also register the work with Hennings...?

In [ ]:
def add_work_to_author(author_qid : str, work_qid : str):
    raise NotImplementedError('Need to investigate if this is wanted')
    work = wbi.item.get(entity_id=author_qid)
    claims_to_add = [
        datatypes.Item(value=work_qid, prop_nr=PROPS['author_of']) 
                     ]
    work.claims.add(claims_to_add)
    work.write()

## Make a new edition entry, referencing the work

In [138]:
def make_edition(author_qid : str, title : str, author : Author, year :int, 
                work_qid : Optional[str], url : Optional[str] = None): 
    language = ENTITIES['German']

    new_edition = wbi.item.new()
    new_edition.labels.set('de', f'{title} (Erstausgabe von {str(year)})')
    # Set a default label too
    new_edition.labels.set('mul', f'{title} (first edition, {str(year)})')

    new_edition.descriptions.set('en', 
                        f'{str(year)} edition of the literary work of fiction by {author.get_name('en')}')
    new_edition.descriptions.set('de', 
                        f'Ausgabe von {str(year)} des fiktionalen literarischen Werks von {author.get_name('de')}')

    new_edition.claims.add([
        datatypes.Item(value=ENTITIES['edition'], prop_nr=PROPS['instance_of']), 
        datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
        datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
        datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
        datatypes.Item(value=language, prop_nr=PROPS['language'])
                            ])   

    if work_qid: 
        new_edition.claims.add([ datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of'])
                               ])
        
    if url:
        new_work.claims.add([
            datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                        ])

    return new_edition


In [139]:
print(author_qid)
print(title)
print(work_qid)
print(author.get_name())
print(url)

Q76815
Die letzte Freude
Q136795200
Emmy Hennings
None


In [140]:
new_edition = make_edition(author_qid = author_qid, year = year,
                           author = author,
                        title = title, 
                        work_qid = work_qid, url=url)
new_edition.get_json()

{'labels': {'de': {'language': 'de',
   'value': 'Die letzte Freude (Erstausgabe von 1913)'},
  'mul': {'language': 'mul',
   'value': 'Die letzte Freude (first edition, 1913)'}},
 'descriptions': {'en': {'language': 'en',
   'value': '1913 edition of the literary work of fiction by Emmy Hennings'},
  'de': {'language': 'de',
   'value': 'Ausgabe von 1913 des fiktionalen literarischen Werks von Emmy Hennings'}},
 'aliases': {},
 'sitelinks': {},
 'type': 'item',
 'claims': {'P31': [{'mainsnak': {'snaktype': 'value',
     'property': 'P31',
     'datatype': 'wikibase-item',
     'datavalue': {'value': {'entity-type': 'item',
       'numeric-id': 3331189,
       'id': 'Q3331189'},
      'type': 'wikibase-entityid'}},
    'type': 'statement',
    'rank': 'normal'}],
  'P1476': [{'mainsnak': {'snaktype': 'value',
     'property': 'P1476',
     'datatype': 'monolingualtext',
     'datavalue': {'value': {'text': 'Die letzte Freude', 'language': 'de'},
      'type': 'monolingualtext'}},
    '

In [141]:
new_edition = new_edition.write()
edition_qid = new_edition.id
edition_qid

'Q136795215'

In [174]:
def make_gutenberg_edition(author_qid : str, title : str, author : Author, source : str,
                           work_qid : Optional[str] = None, edition_qid : Optional[str] = None, 
                           url : Optional[str] = None, year : Optional[int]=None, pg_id : Optional[str] = None): 

    language = ENTITIES['German']

    new_edition = wbi.item.new()

    if source == 'PG-DE':
        gb_edition_descr = {'en': 'Projekt Gutenberg-DE edition', 
                          'de': 'Projekt Gutenberg-DE Edition'}
    elif source == 'PG-US':
        gb_edition_descr = {'en': 'Project Gutenberg edition', 
                          'de': 'Project Gutenberg Edition'}
    else:
        raise Exception(f"source must be one of 'PG-DE' or 'PG-US'")

    new_edition.labels.set('de', title + f' ({gb_edition_descr['de']})')
    # Set a default label too
    new_edition.labels.set('mul', title + f' ({gb_edition_descr['en']})')

    new_edition.descriptions.set('en', 
                f'{gb_edition_descr['en']} of the literary work of fiction by {author.get_name('de')}')
    new_edition.descriptions.set('de', 
                f'{gb_edition_descr['de']} des fiktionalen literarischen Werks von {author.get_name('de')}')

    new_edition.claims.add([
        datatypes.Item(value=ENTITIES['edition'], prop_nr=PROPS['instance_of']), 
        datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of']),
        datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
        datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
        # We're not sure about the year, so let's leave it out
        #datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
        datatypes.Item(value=language, prop_nr=PROPS['language'])
                            ])

    if work_qid: 
        new_edition.claims.add([ datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of'])
                               ])
        
    if edition_qid: 
        new_edition.claims.add([ datatypes.Item(value=edition_qid, prop_nr=PROPS['based_on'])
                               ])   
    if url:
        new_edition.claims.add([
            datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                        ])
        
    if year:
        new_edition.claims.add([
            datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
                        ])
                  # already made sure its a string in the function header, but let's be robust
    if pg_id and (source == 'PG-US'):
            new_edition.claims.add([
                datatypes.ExternalID(value=pg_id, prop_nr=PROPS['project_gb_ebook_id']),
                        ])
        
    return new_edition



In [167]:
source = work['source']
pg_id = work['gutenberg_id']
print(author_qid)
print(title)
print(work_qid)
print(edition_qid)
print(url)
print(pg_id)
print(type(pg_id))
print(source)
print(pg_id and (source == 'PG-US'))

Q76815
Die letzte Freude
Q136795200
Q136795215
None
40218
<class 'str'>
PG-US
True


In [168]:
new_gb_edition = make_gutenberg_edition(author_qid = author_qid, 
                           title = title, 
                           author = author, 
                           work_qid= work_qid,
                           edition_qid = edition_qid, 
                           pg_id=pg_id, source=source)
new_gb_edition.get_json()

{'labels': {'de': {'language': 'de',
   'value': 'Die letzte Freude (Project Gutenberg (US) Edition)'},
  'mul': {'language': 'mul',
   'value': 'Die letzte Freude (Project Gutenberg (US) edition)'}},
 'descriptions': {'en': {'language': 'en',
   'value': 'Project Gutenberg (US) edition of the literary work of fiction by Emmy Hennings'},
  'de': {'language': 'de',
   'value': 'Project Gutenberg (US) Edition des fiktionalen literarischen Werks von Emmy Hennings'}},
 'aliases': {},
 'sitelinks': {},
 'type': 'item',
 'claims': {'P31': [{'mainsnak': {'snaktype': 'value',
     'property': 'P31',
     'datatype': 'wikibase-item',
     'datavalue': {'value': {'entity-type': 'item',
       'numeric-id': 3331189,
       'id': 'Q3331189'},
      'type': 'wikibase-entityid'}},
    'type': 'statement',
    'rank': 'normal'}],
  'P629': [{'mainsnak': {'snaktype': 'value',
     'property': 'P629',
     'datatype': 'wikibase-item',
     'datavalue': {'value': {'entity-type': 'item',
       'numeric-

In [ ]:
new_gb_edition = new_gb_edition.write()
gb_edition_qid = new_gb_edition.id
gb_edition_qid

'Q136795666'

In [171]:
def add_editions_to_work(work_qid : str, edition_qids : list[str]):
    work = wbi.item.get(entity_id=work_qid)
    claims_to_add = [datatypes.Item(value=edition_qid, prop_nr=PROPS['has_edition_or_translation']) 
                     for edition_qid in edition_qids]
    work.claims.add(claims_to_add)
    work.write()


In [173]:
add_editions_to_work(work_qid=work_qid, edition_qids=[edition_qid, gb_edition_qid])

## Find these new works using SPARQL

In [177]:
print(make_sparql_authors_works(author_qid))

SELECT ?item ?title ?year (COUNT(?edition) as ?count)
WHERE {
    # is a literary work
  ?item wdt:P31 wd:Q7725634 .
  # author is kahane
  ?item wdt:P50 wd:Q76815 . 
  OPTIONAL {
  ?edition wdt:P629 ?item . }
  # tile in german
  OPTIONAL {
    ?item wdt:P1476 ?title .} 
  OPTIONAL {
    ?item wdt:P577 ?year . }
  }
GROUP BY ?item ?title ?year
ORDER BY DESC(?count)


### Author now has these editions on wikidata:

In [178]:
result = wbi_helpers.execute_sparql_query(make_sparql_authors_works(author_qid), 
                                          user_agent=wbi_config['USER_AGENT'])
result

{'head': {'vars': ['item', 'title', 'year', 'count']},
 'results': {'bindings': [{'item': {'type': 'uri',
     'value': 'http://www.wikidata.org/entity/Q136794361'},
    'title': {'xml:lang': 'de', 'type': 'literal', 'value': 'Gefängnis'},
    'year': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
     'type': 'literal',
     'value': '1919-01-01T00:00:00Z'},
    'count': {'datatype': 'http://www.w3.org/2001/XMLSchema#integer',
     'type': 'literal',
     'value': '2'}},
   {'item': {'type': 'uri',
     'value': 'http://www.wikidata.org/entity/Q136795200'},
    'title': {'xml:lang': 'de',
     'type': 'literal',
     'value': 'Die letzte Freude'},
    'year': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
     'type': 'literal',
     'value': '1913-01-01T00:00:00Z'},
    'count': {'datatype': 'http://www.w3.org/2001/XMLSchema#integer',
     'type': 'literal',
     'value': '2'}}]}}